# Week 12 Graded Mini Project
# E-Commerce Order, Refund & Exception Handling using CrewAI

## Objective
Build a goal-oriented multi-agent workflow for ecommerce customer support with explicit policy reasoning and escalation decisions. E-commerce platforms handle large volumes of customer queries about delayed deliveries, refunds, damaged products, and return eligibility.

## Architecture

Customer Query
→ Order Issue Identification Agent
→ Policy Interpretation Agent  (returns, refunds, exceptions)
→ Resolution Recommendation Agent
→ Escalation Agent  (manual approval required or not)

All agents depend on outputs from previous agents.


In [1]:
# Install dependencies
# !pip install crewai langchain-openai


In [2]:
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI
import utils

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)


c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:949: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warnings.warn(


## Mock Knowledge Sources (No RAG Required)

In [3]:
MOCK_ORDERS = {
    'ORD1001': {'customer':'John Smith','value':1200,'status':'Delivered','days_since_delivery':2},
    'ORD1002': {'customer':'Alice Brown','value':80,'status':'Delayed','days_since_order':12},
    'ORD1003': {'customer':'Bob Wilson','value':250,'status':'Delivered','days_since_delivery':45},
    'ORD1004': {'customer':'Emma Davis','value':600,'status':'Delivered','days_since_delivery':5},
    'ORD1005': {'customer':'Mike Lee','value':150,'status':'Lost','days_since_order':8}
}


In [4]:
ECOMMERCE_POLICIES  = """
Returns:
- Allowed within 30 days

Refunds:
- Auto refund limit = $500
- Above $500 requires manual review

Damaged Products:
- Replacement allowed
- Refund allowed

Fraud:
- Always escalate
"""

## Agent Definitions

In [5]:
issue_agent = Agent(
    role='Order Issue Identification Agent',
    goal='Classify issue, urgency, sentiment and confidence',
    backstory='Expert ecommerce support analyst',
    verbose=False,
    llm=llm,
    allow_delegation=False
)

policy_agent = Agent(
    role='Policy Interpretation Agent',
    goal='Apply business policies to customer issues',
    backstory='Expert in returns, refunds and exception policies',
    verbose=False,
    llm=llm,
    allow_delegation=False
)

resolution_agent = Agent(
    role='Resolution Recommendation Agent',
    goal='Recommend the best customer resolution',
    backstory='Customer experience specialist',
    verbose=False,
    llm=llm,
    allow_delegation=False
)

escalation_agent = Agent(
    role='Escalation Agent',
    goal='Determine manual review requirement',
    backstory='Risk management specialist',
    verbose=False,
    llm=llm,
    allow_delegation=False
)


## Task Definitions

In [6]:
issue_task = Task(
    description='''Analyze customer query {customer_query}.
    Use order information:
    {order_details}

    Using ONLY the information above, classify the issue.
    Do not search for additional information.
    Do not use external tools.
    ''',
    agent=issue_agent,
    expected_output='Structured issue classification'
)


In [7]:
policy_task = Task(
    description=f'''
    Apply these ecommerce policies:

    {ECOMMERCE_POLICIES}

    Using the issue classification from the previous agent:

    1. Determine eligibility
    2. Determine applicable policy
    3. Determine exception requirements
    4. Provide policy reasoning

    Do not search for additional information.
    Do not use external tools.
    
    ''',
    agent=policy_agent,
    expected_output='Policy evaluation'
)


In [8]:
resolution_task = Task(
    description='''Using previous outputs:

    Generate:
    - Recommended action
    - Customer response
    - Next steps

    Do not search for additional information.
    Do not use external tools.
    
    ''',
    agent=resolution_agent,
    expected_output='Resolution recommendation'
)


In [9]:
escalation_task = Task(
    description=f'''
    Escalation Rules:

    {ECOMMERCE_POLICIES}

    Escalate if:
    - Fraud detected
    - Refund exceeds auto refund limit
    - Return window exceeded
    - Lost shipment investigation required
    - Confidence below 70%
    - High urgency complaint

    Return:
    AUTO_RESOLVE or ESCALATE
    with detailed justification.

    Do not search for additional information.
    Do not use external tools.
    
    ''',
    agent=escalation_agent,
    expected_output='Escalation decision'
)


## Crew Definition

In [10]:
ecommerce_crew = Crew(
    agents=[issue_agent, policy_agent, resolution_agent, escalation_agent],
    tasks=[issue_task, policy_task, resolution_task, escalation_task],
    process=Process.sequential,
    verbose=False
)


## Representative Test Cases

In [11]:
TEST_CASES = [
    {
        'order_id':'ORD1002',
        'query':'My order is delayed by 12 days'
    },
    {
        'order_id':'ORD1003',
        'query':'I want to return a product after 45 days'
    },
    {
        'order_id':'ORD1004',
        'query':'Refund requested for a $600 item'
    },
    {
        'order_id':'ORD1005',
        'query':'Package lost in transit'
    },
    {
        'order_id':'ORD1001',
        'query':'Suspected fraudulent payment activity'
    }
]


In [12]:
for case in TEST_CASES:

    order_details = MOCK_ORDERS[case['order_id']]

    print('='*100)
    print('Order ID:', case['order_id'])
    print('Query:', case['query'])

    result = ecommerce_crew.kickoff(
        inputs={
            'customer_query': case['query'],
            'order_details': order_details
        }
    )

    print(result)


Order ID: ORD1002
Query: My order is delayed by 12 days
Escalate the situation regarding Alice Brown's delayed order due to the high urgency complaint. The delay has caused frustration for the customer, and it is essential to address her concerns promptly. The escalation is justified as it falls under the category of high urgency complaint, which requires immediate attention to ensure customer satisfaction. 

In addition to escalating, I recommend drafting a personalized email to Alice Brown that includes the following elements: 
1. A clear update on her order status and expected delivery date.
2. A sincere apology for the delay and any inconvenience it has caused.
3. An offer of a 10% discount on her next purchase as a gesture of goodwill.

This approach not only addresses the immediate concern but also demonstrates our commitment to customer service. After sending the communication, it is crucial to monitor the situation closely and follow up with Alice after the order is delivered t

## Assignment Requirement Mapping

✓ 4 collaborating agents

✓ Clear role and goal for each agent

✓ Sequential handoffs

✓ Policy reasoning using static knowledge source

✓ Explicit escalation logic

✓ Mock data source

✓ 6 representative test cases

✓ End-to-end workflow demonstration

✓ No RAG used
